In [1]:
from EGPA import *
import numpy as np
import sys
import os
# Get the absolute path of the main directory
main_dir = os.path.abspath(os.path.join(os.getcwd(), "../.."))
# Add the main directory to sys.path
sys.path.append(main_dir)
# importing
from OptFunctions.TwoDBox import *
from Box_functions import *
from TwoD_EGPA import *
import warnings
warnings.filterwarnings('ignore')
import time

from skopt import gp_minimize
from skopt.space import Integer, Real, Categorical
from skopt.utils import use_named_args

#Primero todas las definiciones necesarias para correr el problema, con independencia de los parámetros del algoritmo
x = np.linspace(0, 1, 20)
y = np.linspace(0, 1, 20)
dx = x[-1]-x[-2]
dy = y[-1]-y[-2]

Cartesianstack = None
for i in range(len(x)):
    for j in range(len(y)):
        cartesian = np.array((x[i], y[j]))
        cartesian = np.reshape(cartesian, (1, 2)) 
        if Cartesianstack is None:
            Cartesianstack=cartesian
        else:
            Cartesianstack = np.concatenate([Cartesianstack, cartesian]) 

# Extract frontier points
frontier_mask = (Cartesianstack[:, 0] == 0) | (Cartesianstack[:, 0] == 1) | (Cartesianstack[:, 1] == 0) | (Cartesianstack[:, 1] == 1)
### Crear datos para fijar evidencias en proceso gaussiano
X_init = Cartesianstack[frontier_mask]
Y_init = np.zeros((len(X_init), 1))
init_data = (X_init, Y_init)

THPsi0flat, THPsi0, THEintegrando = Psinxny(1,1, x=x, y=y, dx=dx, dy=dy)
THPsi1flat, THPsi1, THE1integrando = Psinxny(1,2,x=x, y=y, dx=dx, dy=dy)
THPsi12flat, THPsi12, THE12integrando = Psinxny(2,1,x=x, y=y, dx=dx, dy=dy)
THPsi2flat, THPsi2, THE2integrando = Psinxny(2, 2, x=x, y=y, dx=dx, dy=dy)
THPsi3flat, THPsi3, THE3integrando = Psinxny(3, 1, x=x, y=y, dx=dx, dy=dy)
THPsi31flat, THPsi31, THE31integrando = Psinxny(1, 3, x=x, y=y, dx=dx, dy=dy)

######### Esto hará falta cuando se hagan los experimentos y se calcule el error cometido en la predicción del GP a puntos fuera del grid de cálculo #################
# GPx = np.linspace(0, 1, 32)
# GPy = np.linspace(0, 1, 32)
# dGPx = GPx[-1] - GPx[-2]
# dGPy = GPy[-1] - GPy[-2]
# GPstack = None
# for i in range(len(GPx)):
#     for j in range(len(GPy)):
#         cartesian = np.array((GPx[i], GPy[j]))
#         cartesian = np.reshape(cartesian, (1, 2)) 
#         if GPstack is None:
#             GPstack=cartesian
#         else:
#             GPstack = np.concatenate([GPstack, cartesian]) 

# GPTHPsi0flat, GPTHPsi0, GPTHEintegrando = Psinxny(1,1, x=GPx, y=GPy, dx=dGPx, dy=dGPy)
# GPTHPsi1flat, GPTHPsi1, GPTHE1integrando = Psinxny(1,2,x=GPx, y=GPy, dx=dGPx, dy=dGPy)
# GPTHPsi12flat, GPTHPsi12, GPTHE12integrando = Psinxny(2,1,x=GPx, y=GPy, dx=dGPx, dy=dGPy)
# GPTHPsi2flat, GPTHPsi2, GPTHE2integrando = Psinxny(2, 2, x=GPx, y=GPy, dx=dGPx, dy=dGPy)
# GPTHPsi3flat, GPTHPsi3, GPTHE3integrando = Psinxny(3, 1, x=GPx, y=GPy, dx=dGPx, dy=dGPy)
# GPTHPsi31flat, GPTHPsi31, GPTHE31integrando = Psinxny(1, 3, x=GPx, y=GPy, dx=dGPx, dy=dGPy)
################################################################################################################################

# Optimización de parámetros de EGPA
# Define the search space
search_space = [
    Categorical([50, 150, 350, 550, 850, 1000], name="size_gen"),  # Discrete values Integer(50, 1000, name="size_gen") will take integer values in the interval 50, 1000
    #Categorical([10, 30, 60, 100], name="dead_iter"),   # Discrete values Categorical([10, 30, 60, 100], name="dead_iter") was not working because GP cannot handle categorical!!!!
    Real(0.005, 0.15, name="alpha", prior='log-uniform'),                # Continuous values
    Real(0.01, 0.15, name="elite_factor", prior='log-uniform'),        # Continuous values
    Real(0.1, 0.2, name="std_lower_bound", prior='log-uniform'),     # Continuous values
    Integer(2, 10, name="init_kernels")      # Discrete values
]

#Limits for max iter and dead iter during optimization of parameters
max_iter = 70
dead_iter = 10

# Simulated function that runs the optimization and computes RMSE
def evaluate_algorithm(size_gen, alpha, elite_factor, std_lower_bound, init_kernels):
    """
    This function should run the actual optimization algorithm using the given parameters.
    It should return the worst MSE (max MSE across the three optimization problems).
    
    """
    print(f'\nTrying the following configuration of parameters:\nsize_gen:{size_gen}, alpha:{alpha},\nelite_factor:{elite_factor}, std_lower_bound:{std_lower_bound}, init_kernels:{init_kernels}')
    ########################## ESTADO 11 ##########################
    params = {}
    params['dx'] = dx
    params['dy'] = dy
    
    kernel_fail=0
    while True:
        
        try: #values of alpha=0.01, elite = 0,006 with 1000 individuals
            Gaussian = TwoD_EGPA(size_gen=size_gen, grid_x=x, grid_y=y, max_iter=max_iter, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels, std_lower_bound=std_lower_bound ,parallelize=True,init_data=init_data, normalize_solutions=True)
            egparesultground = Gaussian.minimize(TwoDBox, params, exp_from = 300, output_runtime=True)
            Estado = np.reshape(egparesultground.data[-1, :-1, 0], (len(egparesultground.data[-1, :-1, 0]), 1))
            Psi0= reconvert_points_from_cartesian_stack_to_2D(Estado, x=x, y=y)
            E0 = EnergyBox(Psi0, dx=dx, dy=dy)
            MSE0 = min((np.mean((egparesultground.data[-1, :-1, 0] - THPsi0flat)**2),
                        np.mean((-1*egparesultground.data[-1, :-1, 0] - THPsi0flat)**2)))
            print(f'La energía del estado 1,1 teórica es: {Enxny(1,1)},\nEGPA encuentra energía: {E0},\ncon un error respecto de la solución teórica discretizada MSE= {MSE0}')
            break
        except:
            kernel_fail+=1
            if kernel_fail==5:
                print('5º Error fiteando el Kernel. Asignando mal MSE\n\n')
                return 1000000
            else:
                print('Error fiteando el Kernel. Reiniciando algoritmo...\n\n') 
                continue
            
    
    ###################### ESTADO 12 o 21 ##############################
    params = {}
    params['dx'] = dx
    params['dy'] = dy
    params['Psi0'] = Psi0
    params['E0'] = E0
    print(f'La Función de coste debe ser la diferencia de energías al cuadrado aprox\nCon fórmula: {(Enxny(1,2) - Enxny(1,1))**2}, con estado teórico discretizado: {(THE1integrando-THEintegrando)**2}')
    print(f'Función de coste para energía teórica discreta F={TwoDBoxExc1(Psi=THPsi1, dx=dx, dy=dy, Psi0=Psi0, E0=E0)}')
    kernel_fail=0
    while True:
        
        try: #values of alpha=0.01, elite = 0,006 with 1000 individuals
            Gaussian = TwoD_EGPA(size_gen=size_gen, grid_x=x, grid_y=y, max_iter=max_iter, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels, std_lower_bound=std_lower_bound ,parallelize=True,init_data=init_data, normalize_solutions=True)
            egparesultexc1 = Gaussian.minimize(TwoDBoxExc1, params, exp_from = 420, output_runtime=True)
            #Force good solution
            # if egparesultexc1.data[-1, -1, 0] >= 213.8: #antes 208.5
            #     continue
            # else:
            #     break
            Estado = np.reshape(egparesultexc1.data[-1, :-1, 0], (len(egparesultexc1.data[-1, :-1, 0]), 1))
            Psi1= reconvert_points_from_cartesian_stack_to_2D(Estado, x=x, y=y)
            E1 = EnergyBox(Psi1, dx=dx, dy=dy)
            MSE1 = min(np.mean((egparesultexc1.data[-1, :-1, 0] - THPsi1flat)**2),
                        np.mean((-1*egparesultexc1.data[-1, :-1, 0] - THPsi1flat)**2),
                        np.mean((egparesultexc1.data[-1, :-1, 0] - THPsi12flat)**2),
                        np.mean((-1*egparesultexc1.data[-1, :-1, 0] - THPsi12flat)**2))
            print(f'La energía del estado 1,2 teórica es: {Enxny(2,1)},\nEGPA encuentra energía: {E1},\ncon un error respecto de la solución teórica discretizada MSE= {MSE1}')

            break
        except:
            kernel_fail+=1
            if kernel_fail==5:
                print('5º Error fiteando el Kernel. Asignando mal MSE\n\n')
                return 1000000
            else:
                print('Error fiteando el Kernel. Reiniciando algoritmo...\n\n') 
                continue

    ###################### ESTADO ORTOGONAL a 12 o 21 ##############################
    params = {}
    params['dx'] = dx
    params['dy'] = dy
    params['Psi0'] = Psi0
    params['E0'] = E0
    params['Psi1'] = Psi1
    params['E1'] = E1

    kernel_fail=0
    while True:
        
        try:
            Gaussian = TwoD_EGPA(size_gen=size_gen, grid_x=x, grid_y=y, max_iter=max_iter, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels, std_lower_bound=std_lower_bound, parallelize=True,init_data=init_data, normalize_solutions=True)
            egparesultexc12 = Gaussian.minimize(TwoDBoxExc12, params, exp_from = 420, output_runtime=True)

            Estado = np.reshape(egparesultexc12.data[-1, :-1, 0], (len(egparesultexc12.data[-1, :-1, 0]), 1))
            Psi12= reconvert_points_from_cartesian_stack_to_2D(Estado, x=x, y=y)
            E12 = EnergyBox(Psi12, dx=dx, dy=dy)
            MSE12 = min(np.mean((egparesultexc12.data[-1, :-1, 0] - THPsi1flat)**2),
                        np.mean((-1*egparesultexc12.data[-1, :-1, 0] - THPsi1flat)**2),
                        np.mean((egparesultexc12.data[-1, :-1, 0] - THPsi12flat)**2),
                        np.mean((-1*egparesultexc12.data[-1, :-1, 0] - THPsi12flat)**2))
            print(f'La energía del estado 1,2 teórica es: {Enxny(2,1)},\nEGPA encuentra energía: {E12},\ncon un error respecto de la solución teórica discretizada MSE= {MSE12}')
            break
        except:
            kernel_fail+=1
            if kernel_fail==5:
                print('5º Error fiteando el Kernel. Asignando mal MSE\n\n')
                return 1000000
            else:
                print('Error fiteando el Kernel. Reiniciando algoritmo...\n\n') 
                continue
    
    
    ###################### ESTADO 22 ##############################
    params = {}
    params['dx'] = dx
    params['dy'] = dy
    params['Psi0'] = Psi0
    params['E0'] = E0
    params['Psi1'] = Psi1
    params['E1'] = E1
    params['Psi12'] = Psi12
    params['E12'] = E12

    print(f'La función de coste tiene que ser la diferencia de energías al cuadrado {(EnergyBox(Psi=THPsi2, dx=dx,dy=dy) - EnergyBox(Psi=THPsi1, dx=dx,dy=dy))**2}')
    print(f'Función de coste para energía teórica discreta F={TwoDBoxExc2(Psi=THPsi2, dx=dx, dy=dy, Psi0=Psi0, E0=E0, Psi1=Psi1, E1=E1, Psi12=Psi12, E12=E12)}')
    kernel_fail=0
    while True:
        
        try:
            Gaussian = TwoD_EGPA(size_gen=size_gen, grid_x=x, grid_y=y, max_iter=max_iter, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels, std_lower_bound=std_lower_bound ,parallelize=True,init_data=init_data, normalize_solutions=True)
            egparesultexc2 = Gaussian.minimize(TwoDBoxExc2, params, exp_from = 403, output_runtime=True)

            Estado = np.reshape(egparesultexc2.data[-1, :-1, 0], (len(egparesultexc2.data[-1, :-1, 0]), 1))
            Psi2= reconvert_points_from_cartesian_stack_to_2D(Estado, x=x, y=y)
            E2 = EnergyBox(Psi2, dx=dx, dy=dy)
            MSE2 = min(np.mean((egparesultexc2.data[-1, :-1, 0] - THPsi2flat)**2),
                        np.mean((-1*egparesultexc2.data[-1, :-1, 0] - THPsi2flat)**2))
            print(f'La energía del estado 2,2 teórica es: {Enxny(2,2)},\nEGPA encuentra energía: {E2},\ncon un error respecto de la solución teórica discretizada MSE= {MSE2}')
            break

        except:
            kernel_fail+=1
            if kernel_fail==5:
                print('5º Error fiteando el Kernel. Asignando mal MSE\n\n')
                return 1000000
            else:
                print('Error fiteando el Kernel. Reiniciando algoritmo...\n\n') 
                continue

    #Finally the MSE_global is the accumulated MSE between the four first calculated states
    MSEs = [MSE0, MSE1, MSE12, MSE2]
    return np.mean(MSEs) #max(rmse_values)  # Min-Max approach: minimize general MSE among 4 states

global call
call = 0

# Wrapper function for Bayesian Optimization
@use_named_args(search_space)
def objective(**params):
    global call
    call +=1
    print(f'\n##############################################\nBO call number {call} of 50\n##############################################\n\n')
    return evaluate_algorithm(**params)

BO_params = {}

# Run Bayesian Optimization
print(f'\n##############################################\nBO started\n##############################################\n\n')
result = gp_minimize(
    func=objective,      # Objective function
    dimensions=search_space,  # Search space
    n_calls=50,         # Number of function evaluations
    random_state=42,     # For reproducibility
    n_initial_points=12#, n_jobs=-1 #Number of cores to run in parallel while running the lbfgs optimizations over the acquisition function. Valid only when acq_optimizer is set to "lbfgs". Defaults to 1 core. If n_jobs=-1, then number of jobs is set to number of cores.
)

# Best parameters found
best_params = dict(zip(["size_gen", "alpha", "elite_factor", "std_lower_bound", "init_kernels"], result.x))
BO_params['params'] = best_params
print(f'\n##############################################\nBO of EGPA parameters ended\n##############################################\n\n')
print(f"Best EGPA Parameters found:", best_params)
print("Best MSE (Mean Approach):", result.fun)

import pickle
ruta = 'BO_params/BOparameters'
with open(f'{ruta}.pkl', "wb") as file:
    pickle.dump(BO_params, file)

for keys in BO_params.keys():
    print(f'Problema {keys}: {BO_params[keys]}\n')



##############################################
BO started
##############################################



##############################################
BO call number 1 of 50
##############################################



Trying the following configuration of parameters:
size_gen:850, alpha:0.009330940740693842,
elite_factor:0.08260146885196834, std_lower_bound:0.15124109062018282, init_kernels:6
IT:  0 	Best cost:  96.89691522042347
IT:  1 	Best cost:  96.89691522042347
IT:  2 	Best cost:  96.89691522042347
IT:  3 	Best cost:  96.89691522042347
IT:  4 	Best cost:  96.89691522042347
IT:  5 	Best cost:  96.89691522042347
IT:  6 	Best cost:  96.89691522042347
IT:  7 	Best cost:  96.89691522042347
IT:  8 	Best cost:  96.89691522042347
La energía del estado 1,1 teórica es: 9.869604401089358,
EGPA encuentra energía: 9.843623073869878,
con un error respecto de la solución teórica discretizada MSE= 0.003179782430317706
La Función de coste debe ser la diferencia de energías al cuadrado 